# Прогнозирование пиковых нагрузок — полный пайплайн
**ВКР · Турбанов Роман · ДВФУ 2026**

Пайплайн: загрузка данных → EDA → признаки → обучение моделей → сравнение → детекция пиков → рекомендации по репликам

---
**Перед запуском:** задайте `CSV_PATH` в ячейке конфигурации (ячейка 3).

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# Добавляем корень проекта в путь (нужно если notebook в папке проекта)
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
import pandas as pd
import joblib

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('tab10')
print('OK')

In [ ]:
import config
from data_collection.csv_loader import load_csv, load_web_traffic
from preprocessing.feature_engineering import FeatureBuilder, split_train_val_test
from models.xgboost_model import train_xgboost, predict_xgboost, get_confidence_interval, feature_importance
from models.linear_model import train_linear, predict_linear
from models.model_comparison import ModelComparison
from evaluation.metrics import evaluate, safe_mape
from evaluation.peak_detection import PeakDetector
from scaling.scaler import MomentumScaler
print('Импорты: OK')

## 1. Конфигурация
Задайте источник данных. Остальные параметры берутся из `config.py`.

In [ ]:
# ── НАСТРОЙТЕ ЗДЕСЬ ─────────────────────────────────────────────────────────
CSV_PATH       = None        # None → web_traffic.csv; или r'C:\path\to\file.csv'
TIMESTAMP_COL  = None        # None → автоопределение
VALUE_COL      = None        # None → автоопределение (первая числовая колонка)
MONTHS         = 12          # сколько месяцев данных использовать (для web_traffic)
USE_SYNTHETIC  = False       # True → синтетические данные (docker не нужен)
INCLUDE_PROPHET = True       # False → пропустить Prophet (быстрее)
SAVE_DIR       = config.MODEL_SAVE_DIR
# ────────────────────────────────────────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Сохранение моделей и CSV → {os.path.abspath(SAVE_DIR)}')

## 2. Загрузка данных

In [ ]:
if USE_SYNTHETIC:
    from data_collection.synthetic_data import generate_synthetic_traffic
    df = generate_synthetic_traffic(
        days=config.SYNTHETIC_DAYS, freq=config.SYNTHETIC_FREQ,
        base_rps=config.SYNTHETIC_BASE_RPS, noise_std=config.SYNTHETIC_NOISE_STD,
        peak_probability=config.SYNTHETIC_PEAK_PROB,
        peak_multiplier=config.SYNTHETIC_PEAK_MULTIPLIER,
    )
    print('Источник: синтетические данные')
elif CSV_PATH:
    df = load_csv(CSV_PATH, timestamp_col=TIMESTAMP_COL, value_col=VALUE_COL)
    print(f'Источник: {CSV_PATH}')
else:
    df = load_web_traffic(months=MONTHS)
    print('Источник: web_traffic.csv')

print(f'\nСтрок: {len(df)}')
print(f'Период: {df["ds"].iloc[0]}  →  {df["ds"].iloc[-1]}')

step = df['ds'].diff().median()
print(f'Шаг:   {step}')
print(f'\nСтатистика значений:')
df['y'].describe().rename('RPS').to_frame()

## 3. Разведочный анализ (EDA)

In [ ]:
# ── Полный временной ряд ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=False)

# Верх: весь ряд
ax = axes[0]
ax.plot(df['ds'], df['y'], linewidth=0.7, color='steelblue', alpha=0.8)
ax.set_title('Временной ряд (весь датасет)', fontsize=13)
ax.set_ylabel('RPS')

# Низ: скользящее среднее
ax2 = axes[1]
window = max(1, len(df) // 100)
ax2.plot(df['ds'], df['y'], linewidth=0.5, color='steelblue', alpha=0.4, label='Факт')
ax2.plot(df['ds'], df['y'].rolling(window).mean(), linewidth=1.5, color='#e74c3c',
         label=f'Скользящее среднее (окно={window})')
ax2.fill_between(df['ds'],
                  df['y'].rolling(window).mean() - df['y'].rolling(window).std(),
                  df['y'].rolling(window).mean() + df['y'].rolling(window).std(),
                  alpha=0.15, color='#e74c3c', label='±1σ')
ax2.set_title('Тренд и волатильность', fontsize=13)
ax2.set_ylabel('RPS')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_timeseries.png'), dpi=150)
plt.show()

In [ ]:
# ── Распределение и выбросы ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Гистограмма
axes[0].hist(df['y'], bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(df['y'].mean(), color='red', linewidth=1.5, label=f'Mean={df["y"].mean():.0f}')
axes[0].axvline(df['y'].median(), color='orange', linewidth=1.5, linestyle='--',
                label=f'Median={df["y"].median():.0f}')
axes[0].set_title('Распределение RPS')
axes[0].set_xlabel('RPS')
axes[0].legend(fontsize=9)

# Box plot
axes[1].boxplot(df['y'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Box plot (выбросы)')
axes[1].set_ylabel('RPS')
axes[1].set_xticks([])

# Квантили
quantiles = [50, 75, 90, 95, 99]
q_vals = [np.percentile(df['y'].dropna(), q) for q in quantiles]
axes[2].barh([f'P{q}' for q in quantiles], q_vals, color='steelblue', alpha=0.7)
for i, v in enumerate(q_vals):
    axes[2].text(v * 1.01, i, f'{v:.0f}', va='center', fontsize=9)
axes[2].set_title('Перцентили RPS')
axes[2].set_xlabel('RPS')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_distribution.png'), dpi=150)
plt.show()

In [ ]:
# ── Суточная и недельная сезонность ─────────────────────────────────────────
df['hour']        = pd.to_datetime(df['ds']).dt.hour
df['day_of_week'] = pd.to_datetime(df['ds']).dt.dayofweek
days_labels = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Суточный профиль
hourly = df.groupby('hour')['y'].agg(['mean', 'std'])
axes[0].plot(hourly.index, hourly['mean'], color='steelblue', linewidth=2)
axes[0].fill_between(hourly.index,
                      hourly['mean'] - hourly['std'],
                      hourly['mean'] + hourly['std'],
                      alpha=0.2, color='steelblue')
axes[0].set_title('Суточный профиль (среднее ± σ)')
axes[0].set_xlabel('Час суток')
axes[0].set_ylabel('RPS')
axes[0].set_xticks(range(0, 24, 3))

# Дневной профиль
daily = df.groupby('day_of_week')['y'].mean()
bars = axes[1].bar([days_labels[i] for i in daily.index], daily.values,
                    color=['#e74c3c' if i >= 5 else 'steelblue' for i in daily.index],
                    alpha=0.8)
axes[1].set_title('Средний RPS по дням недели')
axes[1].set_ylabel('RPS')

# Тепловая карта: час × день недели
pivot = df.groupby(['day_of_week', 'hour'])['y'].mean().unstack()
pivot.index = [days_labels[i] for i in pivot.index]
sns.heatmap(pivot, ax=axes[2], cmap='YlOrRd', fmt='.0f',
            linewidths=0.3, cbar_kws={'label': 'RPS'})
axes[2].set_title('Тепловая карта: день × час')
axes[2].set_xlabel('Час')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_seasonality.png'), dpi=150)
plt.show()

# Удаляем временные колонки
df = df[['ds', 'y']]

## 4. Формирование признаков (Feature Engineering)

In [ ]:
builder = FeatureBuilder()
sample_features = builder.transform(df.head(500)).dropna()

print(f'Количество признаков: {len(builder.FEATURE_COLS)}')
print(f'\nСписок признаков:')
for i, col in enumerate(builder.FEATURE_COLS, 1):
    print(f'  {i:2d}. {col}')

print(f'\nПример матрицы признаков (первые 3 строки):')
sample_features[builder.FEATURE_COLS].head(3)

In [ ]:
# Корреляция признаков с целевой переменной y
corr_data = sample_features[builder.FEATURE_COLS + ['y']].corr()['y'].drop('y').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#27ae60' if v > 0 else '#e74c3c' for v in corr_data.values]
ax.barh(corr_data.index[::-1], corr_data.values[::-1], color=colors[::-1], alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Корреляция признаков с RPS (целевой переменной)')
ax.set_xlabel('Коэффициент корреляции Пирсона')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'feature_correlations.png'), dpi=150)
plt.show()

## 5. Разбиение на обучающую / валидационную / тестовую выборки

In [ ]:
n = len(df)
TEST_PERIODS = min(480, n // 6)
VAL_PERIODS  = TEST_PERIODS

train, val, test = split_train_val_test(df, test_hours=TEST_PERIODS, val_hours=VAL_PERIODS)

print(f'Train : {len(train):5d} точек  ({train["ds"].iloc[0].date()} – {train["ds"].iloc[-1].date()})')
print(f'Val   : {len(val):5d} точек  ({val["ds"].iloc[0].date()} – {val["ds"].iloc[-1].date()})')
print(f'Test  : {len(test):5d} точек  ({test["ds"].iloc[0].date()} – {test["ds"].iloc[-1].date()})')
print(f'\nДоли: train={len(train)/n:.0%}  val={len(val)/n:.0%}  test={len(test)/n:.0%}')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(train['ds'], train['y'], color='steelblue', linewidth=0.7, label=f'Train ({len(train)} точек)')
ax.plot(val['ds'],   val['y'],   color='orange',    linewidth=0.9, label=f'Validation ({len(val)} точек)')
ax.plot(test['ds'],  test['y'],  color='#27ae60',   linewidth=1.1, label=f'Test ({len(test)} точек)')

ax.axvspan(val['ds'].iloc[0], val['ds'].iloc[-1], alpha=0.07, color='orange')
ax.axvspan(test['ds'].iloc[0], test['ds'].iloc[-1], alpha=0.10, color='green')

ax.set_title('Хронологическое разбиение выборок (без перемешивания)', fontsize=13)
ax.set_ylabel('RPS')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'split_visualization.png'), dpi=150)
plt.show()

## 6. Обучение моделей

In [ ]:
# Строим признаки на объединённых данных (context-aware)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = builder.transform_splits(train, val, test)
print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')

In [ ]:
# ── Baseline: Ridge Regression ───────────────────────────────────────────────
print('=== Linear Regression (baseline) ===')
model_lr = train_linear(X_train, y_train)
preds_lr = predict_linear(model_lr, X_test)
metrics_lr = evaluate(y_test.values, preds_lr, 'LinearRegression', verbose=True)
joblib.dump(model_lr, os.path.join(SAVE_DIR, 'linear.pkl'))

In [ ]:
# ── XGBoost ──────────────────────────────────────────────────────────────────
print('=== XGBoost ===')
model_xgb = train_xgboost(
    X_train, y_train, X_val, y_val,
    n_estimators=config.XGB_N_ESTIMATORS,
    max_depth=config.XGB_MAX_DEPTH,
    learning_rate=config.XGB_LEARNING_RATE,
    early_stopping_rounds=config.XGB_EARLY_STOPPING,
    save_path=os.path.join(SAVE_DIR, 'xgboost.pkl'),
)
preds_xgb = predict_xgboost(model_xgb, X_test)
metrics_xgb = evaluate(y_test.values, preds_xgb, 'XGBoost', verbose=True)

In [ ]:
# ── Prophet (опционально) ────────────────────────────────────────────────────
preds_prophet = None
metrics_prophet = None

if INCLUDE_PROPHET:
    try:
        from models.prophet_model import train_prophet, predict_prophet
        print('=== Prophet ===')
        train_val = pd.concat([train, val]).sort_values('ds')
        model_prophet = train_prophet(
            train_val, val,
            save_path=os.path.join(SAVE_DIR, 'prophet.pkl'),
        )
        freq_str = pd.infer_freq(df['ds']) or 'h'
        prophet_df = predict_prophet(model_prophet, periods=len(test), freq=freq_str)
        preds_prophet = prophet_df['yhat'].values[:len(y_test)]
        preds_prophet = np.clip(preds_prophet, 0, None)
        metrics_prophet = evaluate(y_test.values, preds_prophet, 'Prophet', verbose=True)
    except Exception as e:
        print(f'Prophet пропущен: {e}')
else:
    print('Prophet пропущен (INCLUDE_PROPHET=False)')

## 7. Сравнение моделей

In [ ]:
all_metrics = {
    'LinearRegression': metrics_lr,
    'XGBoost': metrics_xgb,
}
all_preds = {
    'LinearRegression': preds_lr,
    'XGBoost': preds_xgb,
}
if metrics_prophet is not None:
    all_metrics['Prophet'] = metrics_prophet
    all_preds['Prophet'] = preds_prophet

metrics_df = pd.DataFrame(all_metrics).T[['MAE', 'RMSE', 'MAPE']]
metrics_df.index.name = 'Модель'

best_model_name = metrics_df['MAPE'].idxmin()
metrics_df['Лучшая'] = metrics_df.index.map(lambda x: '✓' if x == best_model_name else '')

print(f'\nИтоговая таблица метрик:')
print(f'{"Модель":<22} {"MAE":>10} {"RMSE":>10} {"MAPE%":>8}')
print('-' * 55)
for name, row in metrics_df.iterrows():
    mark = ' [*]' if row['Лучшая'] == '✓' else '    '
    print(f'{name+mark:<26} {row["MAE"]:>10.2f} {row["RMSE"]:>10.2f} {row["MAPE"]:>7.2f}%')
print('-' * 55)
print(f'[*] Лучшая модель по MAPE: {best_model_name}')

In [ ]:
# ── Bar chart метрик ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
model_names = list(all_metrics.keys())

for i, metric in enumerate(['MAE', 'RMSE', 'MAPE']):
    vals = [all_metrics[m][metric] for m in model_names]
    bars = axes[i].bar(model_names, vals, color=colors[:len(model_names)], alpha=0.85)
    best_idx = vals.index(min(vals))
    bars[best_idx].set_edgecolor('black')
    bars[best_idx].set_linewidth(2.5)
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                     f'{v:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    unit = '%' if metric == 'MAPE' else ''
    axes[i].set_title(f'{metric}{unit}', fontsize=12)
    axes[i].set_ylabel(f'{metric}{unit}')
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Сравнение моделей (меньше = лучше)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'comparison_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Все прогнозы на одном графике (тест-период) ──────────────────────────────
plot_colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
test_ds = test['ds'].values[:len(y_test)]

fig, axes = plt.subplots(2, 1, figsize=(15, 9))

for ax, zoom in zip(axes, [False, True]):
    if zoom:
        ax.plot(test_ds, y_test.values, color='black', linewidth=2,
                label='Факт', zorder=10)
        for (name, preds), color in zip(all_preds.items(), plot_colors):
            n = min(len(test_ds), len(preds))
            ax.plot(test_ds[:n], preds[:n], linewidth=1.5, linestyle='--',
                    color=color, label=name, alpha=0.9)
        ax.set_title('Сравнение прогнозов: тест-период (zoom)', fontsize=12)
    else:
        ax.plot(train['ds'], train['y'], color='steelblue', linewidth=0.5,
                alpha=0.5, label='Train')
        ax.plot(val['ds'], val['y'], color='orange', linewidth=0.7,
                alpha=0.7, label='Validation')
        ax.plot(test['ds'], y_test.values, color='black', linewidth=1.5,
                label='Test (факт)', zorder=10)
        for (name, preds), color in zip(all_preds.items(), plot_colors):
            n = min(len(test['ds']), len(preds))
            ax.plot(test['ds'].values[:n], preds[:n], linewidth=1.5,
                    linestyle='--', color=color, label=name, alpha=0.9)
        ax.axvspan(test['ds'].iloc[0], test['ds'].iloc[-1], alpha=0.07, color='green')
        ax.set_title('Сравнение прогнозов: полный ряд', fontsize=12)
    ax.set_ylabel('RPS')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'comparison_all_models.png'), dpi=150)
plt.show()

## 8. Доверительный интервал XGBoost (квантильная регрессия)

In [ ]:
print('Обучение квантильных моделей (q=0.1 и q=0.9)...')
lower, upper = get_confidence_interval(
    X_train, y_train, X_val, y_val, X_test,
    save_dir=SAVE_DIR,
)
# Оценка покрытия
coverage = np.mean((y_test.values >= lower) & (y_test.values <= upper))
print(f'Покрытие ДИ 80%: {coverage:.1%} (идеал ≈ 80%)')

In [ ]:
n = min(len(test['ds']), len(preds_xgb), len(lower), len(upper))
test_ds_plot = test['ds'].values[:n]

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(test_ds_plot, y_test.values[:n], color='#27ae60', linewidth=2,
        label='Факт', zorder=5)
ax.plot(test_ds_plot, preds_xgb[:n], color='#e74c3c', linewidth=1.8,
        linestyle='--', label='XGBoost (прогноз)')
ax.fill_between(test_ds_plot, lower[:n], upper[:n],
                alpha=0.25, color='#e74c3c', label=f'ДИ 80% (покрытие={coverage:.0%})')
ax.set_title('XGBoost: прогноз с доверительным интервалом 80%', fontsize=13)
ax.set_ylabel('RPS')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'xgboost_confidence_interval.png'), dpi=150)
plt.show()

## 9. Важность признаков

In [ ]:
imp_df = feature_importance(model_xgb, feature_names=list(X_train.columns))
print('Важность признаков XGBoost (gain):')
print(imp_df.to_string(index=False))
imp_df.to_csv(os.path.join(SAVE_DIR, 'feature_importance.csv'), index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
norm = imp_df['importance'] / imp_df['importance'].sum() * 100
bar_colors = plt.cm.RdYlGn(norm.values / norm.max())
bars = ax.barh(imp_df['feature'][::-1], norm[::-1], color=bar_colors[::-1], alpha=0.85)
for bar, v in zip(bars, norm[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}%', va='center', fontsize=9)
ax.set_title('Важность признаков XGBoost (gain, %)', fontsize=13)
ax.set_xlabel('Важность (%)')
ax.set_xlim(0, norm.max() * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'feature_importance.png'), dpi=150)
plt.show()

## 10. Детекция пиковых нагрузок

In [ ]:
rps_max = float(train['y'].max())
target_per_replica = rps_max / config.MAX_REPLICAS

detector = PeakDetector(
    method=config.PEAK_METHOD,
    k=config.PEAK_K,
    target_rps_per_replica=target_per_replica,
    min_replicas=config.MIN_REPLICAS,
    max_replicas=config.MAX_REPLICAS,
    warning_ratio=config.ALERT_WARNING_RATIO,
    critical_ratio=config.ALERT_CRITICAL_RATIO,
)
detector.fit(train['y'])
print(f'Порог: {detector.threshold:.2f} RPS  (метод: {config.PEAK_METHOD}, k={config.PEAK_K})')

predicted_series = pd.Series(preds_xgb[:len(y_test)], index=y_test.index)
events_df = detector.detect_series(y_test, predicted_series)

summary = detector.summary(events_df)
print(f'\nСводка:')
print(f'  Всего точек:    {summary["total_points"]}')
print(f'  Пиков:          {summary["peaks_detected"]} ({summary["peak_ratio_pct"]}%)')
print(f'  По severity:    {summary["severity_counts"]}')

In [ ]:
# ── Временной ряд с подсвеченными пиками ────────────────────────────────────
severity_colors = {'ok': None, 'info': '#3498db', 'warning': '#f39c12', 'critical': '#e74c3c'}

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

ax = axes[0]
ax.plot(events_df['timestamp'], events_df['rps'], color='#27ae60', linewidth=1.5,
        label='Факт', zorder=5)
ax.plot(events_df['timestamp'], events_df['predicted'], color='#e74c3c', linewidth=1.5,
        linestyle='--', label='Прогноз XGBoost', alpha=0.8)
ax.axhline(detector.threshold, color='black', linewidth=1.5, linestyle=':',
           label=f'Порог ({detector.threshold:.0f} RPS)', zorder=8)
ax.axhline(detector.threshold * config.ALERT_WARNING_RATIO, color='#f39c12',
           linewidth=1, linestyle=':', alpha=0.7,
           label=f'Warning ({config.ALERT_WARNING_RATIO*100:.0f}%)')

# Подсвечиваем пики
for _, ev in events_df[events_df['is_peak']].iterrows():
    color = severity_colors.get(ev['severity'])
    if color:
        ax.axvline(ev['timestamp'], color=color, alpha=0.4, linewidth=1.5)

ax.set_title('Детекция пиков: факт vs прогноз', fontsize=13)
ax.set_ylabel('RPS')
ax.legend(fontsize=9, ncol=2)

# Нижний: уровни severity
ax2 = axes[1]
sev_map = {'ok': 0, 'info': 1, 'warning': 2, 'critical': 3}
sev_colors_list = [severity_colors.get(s) or '#95a5a6' for s in events_df['severity']]
ax2.scatter(events_df['timestamp'], events_df['severity'].map(sev_map),
            c=sev_colors_list, s=8, alpha=0.7)
ax2.set_yticks([0, 1, 2, 3])
ax2.set_yticklabels(['ok', 'info', 'warning', 'critical'])
ax2.set_title('Уровень alert по времени', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'peak_detection.png'), dpi=150)
plt.show()

In [ ]:
# ── Топ-10 пиков ─────────────────────────────────────────────────────────────
peaks = events_df[events_df['is_peak']].copy()
peaks['exceeding_pct'] = ((peaks['predicted'] - detector.threshold) / detector.threshold * 100).round(1)
print(f'Первые 10 пиков на тестовом периоде:')
peaks[['timestamp', 'rps', 'predicted', 'severity', 'recommended_replicas', 'exceeding_pct']].head(10)

## 11. Рекомендации по масштабированию реплик

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)

ax = axes[0]
ax.plot(events_df['timestamp'], events_df['predicted'], color='#e74c3c',
        linewidth=1.5, label='Прогноз RPS')
ax.axhline(detector.threshold, color='black', linewidth=1.2, linestyle=':',
           label=f'Порог ({detector.threshold:.0f})')
ax.set_ylabel('Прогноз RPS')
ax.set_title('Проактивное масштабирование: прогноз → рекомендации', fontsize=13)
ax.legend(fontsize=9)

ax2 = axes[1]
replica_colors = events_df['severity'].map(
    {'ok': '#27ae60', 'info': '#3498db', 'warning': '#f39c12', 'critical': '#e74c3c'}
)
ax2.step(events_df['timestamp'], events_df['recommended_replicas'],
         color='steelblue', linewidth=2, where='post', label='Реплик')
ax2.fill_between(events_df['timestamp'], events_df['recommended_replicas'],
                  step='post', alpha=0.2, color='steelblue')
ax2.axhline(config.MIN_REPLICAS, color='green', linewidth=1, linestyle='--',
            alpha=0.7, label=f'Min={config.MIN_REPLICAS}')
ax2.axhline(config.MAX_REPLICAS, color='red', linewidth=1, linestyle='--',
            alpha=0.7, label=f'Max={config.MAX_REPLICAS}')
ax2.set_ylabel('Число реплик')
ax2.set_yticks(range(config.MIN_REPLICAS, config.MAX_REPLICAS + 1))
ax2.set_ylim(0, config.MAX_REPLICAS + 1)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'scaling_recommendations.png'), dpi=150)
plt.show()

print(f'\nСредние рекомендации по репликам:')
print(events_df['recommended_replicas'].value_counts().sort_index())

## 12. Экспорт результатов

In [ ]:
# ── CSV: метрики ─────────────────────────────────────────────────────────────
metrics_out = pd.DataFrame(all_metrics).T[['MAE', 'RMSE', 'MAPE']]
metrics_out.index.name = 'model'
metrics_out.to_csv(os.path.join(SAVE_DIR, 'metrics_comparison.csv'))

# ── CSV: предсказания ────────────────────────────────────────────────────────
n_out = min(len(test['ds']), len(preds_xgb), len(lower), len(upper))
pred_df = test[['ds', 'y']].iloc[:n_out].copy().reset_index(drop=True)
pred_df['XGBoost'] = preds_xgb[:n_out]
pred_df['XGBoost_lower'] = lower[:n_out]
pred_df['XGBoost_upper'] = upper[:n_out]
pred_df['LinearRegression'] = preds_lr[:n_out]
if preds_prophet is not None:
    pred_df['Prophet'] = preds_prophet[:n_out]
pred_df.to_csv(os.path.join(SAVE_DIR, 'predictions.csv'), index=False)

# ── CSV: события детекции пиков ───────────────────────────────────────────────
events_df.to_csv(os.path.join(SAVE_DIR, 'peak_events.csv'), index=False)

# ── CSV: важность признаков ───────────────────────────────────────────────────
imp_df.to_csv(os.path.join(SAVE_DIR, 'feature_importance.csv'), index=False)

print(f'Все файлы сохранены в: {os.path.abspath(SAVE_DIR)}/')
print()
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f'  {f:<40} {size/1024:6.1f} KB')

In [ ]:
# ── Итоговая сводка ──────────────────────────────────────────────────────────
print('=' * 55)
print('ИТОГОВАЯ СВОДКА')
print('=' * 55)
print(f'Данных:          {len(df)} точек')
print(f'Период:          {df["ds"].iloc[0].date()} – {df["ds"].iloc[-1].date()}')
print(f'Шаг:             {step}')
print(f'Признаков:       {X_train.shape[1]}')
print()
print(f'Лучшая модель:   {best_model_name}')
print(f'  MAE  = {all_metrics[best_model_name]["MAE"]:.2f}')
print(f'  RMSE = {all_metrics[best_model_name]["RMSE"]:.2f}')
print(f'  MAPE = {all_metrics[best_model_name]["MAPE"]:.2f}%')
print()
print(f'Порог пика:      {detector.threshold:.2f} RPS')
print(f'Пиков в тесте:   {summary["peaks_detected"]} / {summary["total_points"]} ({summary["peak_ratio_pct"]}%)')
print(f'Покрытие ДИ 80%: {coverage:.1%}')
print('=' * 55)